In [0]:
from pyspark.sql import functions as F

SILVER       = "capstone_project_dev.silver.validated_metadata"
GOLD_METRICS = "capstone_project_dev.gold.governance_metrics"
GOLD_NONCOMPLIANT = "capstone_project_dev.gold.non_compliant_tables"

print("Setup done")

In [0]:
silver = spark.table(SILVER)

print(f"Rows loaded : {silver.count():,}")
print(f"Cols loaded : {len(silver.columns)}")

In [0]:
total_tables  = silver.select("table_name").distinct().count()
total_columns = silver.select("table_name", "column_name").distinct().count()

# Structural compliance — tables with column_count >= threshold (mean - 1 stddev = 38.6)
structural_compliance_pct = (
    silver.select("table_name", "rule06_below_col_standard")
          .distinct()
          .filter(F.col("rule06_below_col_standard") == False)
          .count()
    / total_tables * 100
)

# PII coverage — PII rows correctly marked Confidential
pii_total = silver.filter(F.col("pii_flag") == True).count()
pii_covered = silver.filter(
    (F.col("pii_flag") == True) &
    (F.col("security_classification") == "Confidential")
).count()
pii_coverage_pct = (pii_covered / pii_total * 100) if pii_total > 0 else 0

# Certification coverage — distinct columns with non-null certification
cert_covered = silver.select("table_name", "column_name", "certification_level") \
    .distinct() \
    .filter(F.col("certification_level").isNotNull()) \
    .count()
cert_coverage_pct = (cert_covered / total_columns * 100)

# CDEs missing a steward
cde_missing_steward = silver.filter(
    (F.col("critical_data_element_flag") == True) &
    (F.col("data_steward").isNull())
).count()

# Metadata completeness — across 4 key fields, deduplicated
fields_to_check = ["column_desc", "term_name", "data_steward", "security_classification"]
total_possible  = total_columns * len(fields_to_check)
total_filled    = sum(
    silver.select("table_name", "column_name", f)
          .distinct()
          .filter(F.col(f).isNotNull())
          .count()
    for f in fields_to_check
)
metadata_completeness_pct = (total_filled / total_possible * 100)

# Overall compliance — deduplicated rows
compliant_rows = silver.select("table_name", "column_name", "compliance_flag") \
    .distinct() \
    .filter(F.col("compliance_flag") == "COMPLIANT") \
    .count()
non_compliant_rows = total_columns - compliant_rows
overall_compliance_pct = (compliant_rows / total_columns * 100)

print(f"Total tables              : {total_tables}")
print(f"Total columns             : {total_columns:,}")
print(f"Structural compliance %   : {structural_compliance_pct:.1f}%")
print(f"PII coverage %            : {pii_coverage_pct:.1f}%")
print(f"Certification coverage %  : {cert_coverage_pct:.1f}%")
print(f"CDE missing steward       : {cde_missing_steward:,}")
print(f"Metadata completeness %   : {metadata_completeness_pct:.1f}%")
print(f"Overall compliance %      : {overall_compliance_pct:.1f}%")